In [2]:
import json
import boto3
from botocore.exceptions import ClientError


def iam_role():
    # Initializing IAM client
    iam_client = boto3.client('iam')


    role_name = 'glue-Stream-Execution-Role'

    # Assume role trust policy for Glue
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "glue.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }

    # Managed policies to attach
    managed_policies = [
        'arn:aws:iam::aws:policy/AmazonS3FullAccess',
        'arn:aws:iam::aws:policy/AmazonKinesisReadOnlyAccess',
        'arn:aws:iam::aws:policy/CloudWatchLogsFullAccess',
        'arn:aws:iam::aws:policy/AWSGlueConsoleFullAccess'
    ]

    # 1. Create the IAM Role
    try:
        response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='IAM role for AWS Glue to read Kinesis, write to S3, and log to CloudWatch'
        )
        print(f"Role '{role_name}' created successfully.")
    except ClientError as e:
        print(f"Error creating role: {e}")
        

    # 2. Attach managed policies
    for policy_arn in managed_policies:
        try:
            iam_client.attach_role_policy(
                RoleName=role_name,
                PolicyArn=policy_arn
            )
            print(f"Attached policy: {policy_arn}")
        except ClientError as e:
            print(f"Error attaching policy {policy_arn}: {e}")


In [3]:


import boto3
from botocore.exceptions import ClientError




def s3_bkt_creation():

    s3_client = boto3.client('s3', region_name='us-east-1')


    bucket_name = 'data-forge-bkt1'
    file_path = 'mobile-logs-generated.csv'
    object_key = 'raw/mobile-logs-generated'
    folders = ['raw/', 'processed/', 'scripts/', 'checkpoint/', 'query-results/','PathToJar/']

    script_file_path = 'script.py'
    script_key = 'scripts/script'

    jar_file_path= 'spark-streaming-kinesis-asl_2.12-3.5.5.jar'
    jar_key = 'PathToJar/spark-streaming-kinesis-asl_2.12-3.5.5.jar'


    try:
        # Creating Bucket
        s3_client.create_bucket(Bucket=bucket_name)
        print(f"Bucket '{bucket_name}' created.")
        
        # Adding Folder 
        for folder in folders:
            s3_client.put_object(Bucket=bucket_name, Key=folder)
            print(f"Folder '{folder}' created.")
        
        # Uploading Mobile logs file    
        s3_client.upload_file(file_path, bucket_name, object_key)
        print(f"File '{file_path}' uploaded to s3://{bucket_name}/{object_key}")

        #uploading script file
        s3_client.upload_file(script_file_path, bucket_name, script_key)
        print(f"File '{script_file_path}' uploaded to s3://{bucket_name}/{script_key}")


         #uploading jar file
        s3_client.upload_file(jar_file_path, bucket_name, jar_key )
        print(f"File '{jar_file_path}' uploaded to s3://{bucket_name}/{jar_key}")

            

    except ClientError as e:
        print(f"Error creating bucket: {e}")






In [4]:
import boto3
import time
from botocore.exceptions import ClientError


def kinesis():

    kinesis_client = boto3.client('kinesis', region_name='us-east-1')

    stream_name = 'kinesis-mobile-log-stream'
    shard_count = 1


    try:
        response = kinesis_client.create_stream(
            StreamName=stream_name,
            ShardCount=shard_count
        )
        print(f"Creating Kinesis stream '{stream_name}' with {shard_count} shard(s)...")
        
    except ClientError as e:
        print(f"Error creating stream: {e}")
        


In [5]:
import boto3
import time
from botocore.exceptions import ClientError

def kinesis_check():
    # Check kinesis Stream status

    kinesis_client = boto3.client('kinesis', region_name='us-east-1')

    stream_name = 'kinesis-mobile-log-stream'

    response = kinesis_client.describe_stream(StreamName=stream_name)
    status = response['StreamDescription']['StreamStatus']
    print(f"Stream status is '{status}'")

In [6]:

import boto3
from botocore.exceptions import ClientError

def create_db():
    # Initialize Glue client
    glue = boto3.client('glue')

    # Define parameters

    bucket_name = 'data-forge-bkt1'
    database_name = "mobile_logs_db"
    crawler_name = "crawler-mobile-processed"
    role_name = "glue-Stream-Execution-Role"
    s3_path = f"s3://{bucket_name}/processed/"

    # Step 1: Create Database
    try:
        print(f"Creating database '{database_name}'...")
        response = glue.create_database(
            DatabaseInput={
                'Name': database_name,
                'Description': 'Database for processed mobile logs'
            }
        )
        print("Database created successfully.")
    except ClientError as e:
        print(f" Failed to create database: {e}")


    

In [7]:
import boto3


def glue_job():
    # Initialize Glue client
    glue = boto3.client('glue')

    # Define parameters
    job_name = 'DataForgeGlueJob1'
    role_name = 'glue-Stream-Execution-Role'
    script_s3_path = 's3://data-forge-bkt1/scripts/script'
    dependent_jars = 's3://data-forge-bkt1/PathToJar/spark-streaming-kinesis-asl_2.12-3.5.5.jar'

    try:
        response = glue.create_job(
            Name=job_name,
            Role=role_name,
            ExecutionProperty={
                'MaxConcurrentRuns': 1
            },
            Command={
                'Name': 'glueetl',
                'ScriptLocation': script_s3_path,
                'PythonVersion': '3'
            },
            DefaultArguments={
                '--extra-jars': dependent_jars
            },
            GlueVersion='5.0',
            Description='Streaming Glue job for DataForge project',
            Tags={
                'Project': 'DataForge',
                'Type': 'StreamingJob'
            }
        )
        
        print(f"Glue job '{job_name}' created successfully!")
        print("Response:", response)

    except glue.exceptions.AlreadyExistsException:
        print(f"Glue job '{job_name}' already exists.")

    except Exception as e:
        print("Error creating Glue job:", str(e))


In [8]:
import csv, json, time, boto3

def producer():

    STREAM_NAME = "kinesis-mobile-log-stream"
    CSV_PATH = "mobile-logs-generated.csv"  
    SLEEP = 0.05

    kinesis = boto3.client("kinesis", region_name="us-east-1")

    with open(CSV_PATH, 'r') as f:

        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            data = json.dumps(row)
            kinesis.put_record(StreamName=STREAM_NAME, Data=data.encode('utf-8'), PartitionKey=str(i%10))
            if i % 100 == 0:
                print("sent", i)
            time.sleep(SLEEP)
            #print(data)



In [ ]:
### from credentials_config import credentials_config
# from s3_bkt_creation import s3_bkt_creation
# from kinesis import kinesis
# from kinesis_check import kinesis_check
# from glue_job import glue_job
# from create_db import create_db
# from iam_role import iam_role
# from producer import producer
# import time


if __name__ == "__main__":
    print("Starting AWS DataForge Pipeline Setup...\n")
    
    iam_role()
    time.sleep(5)
    s3_bkt_creation()
    time.sleep(5)
    kinesis()
    time.sleep(5)
    kinesis_check()
    time.sleep(5)
    create_db()
    time.sleep(5)
    glue_job()
    time.sleep(5)
    producer()

    

    print("\n All steps completed successfully!")


In [10]:
producer()

sent 0
sent 100
sent 200
sent 300
sent 400
sent 500
sent 600
sent 700
sent 800
sent 900
sent 1000
sent 1100
sent 1200
sent 1300
sent 1400
sent 1500
sent 1600
sent 1700
sent 1800


ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the PutRecord operation: Stream kinesis-mobile-log-stream under account 801341413974 not found.